In [17]:
import sys
import random
import shutil
from pathlib import Path
import csv
print(f"Python version: {sys.version}")

Python version: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]


In [18]:
%cd /home/jupyter/HocTap/son/Nhom11_UniMERNet

/home/jupyter/HocTap/son/Nhom11_UniMERNet


In [3]:
# Khai báo các hàm


def export_split(samples, out_img_dir, out_txt_path, out_csv_path):
    lines = []
    mapping_rows = []

    for new_idx, (src_path, old_idx) in enumerate(samples):
        new_name = f"{new_idx:07d}.png"
        dst_path = out_img_dir / new_name

        shutil.copy2(src_path, dst_path)

        formula = labels[old_idx]
        lines.append(formula)
        mapping_rows.append([new_idx, new_name, src_path.name, old_idx, formula])

    out_txt_path.write_text("\n".join(lines), encoding="utf-8")

    with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "new_index",
            "new_image_name",
            "original_image_name",
            "original_label_index",
            "formula"
        ])
        writer.writerows(mapping_rows)

    print(f"Saved {len(samples)} samples to {out_img_dir}")
    print(f"Saved labels to {out_txt_path}")
    print(f"Saved mapping to {out_csv_path}")


In [4]:
src_img_dir = Path("/home/jupyter/HocTap/son/data_train/UniMER-1M/images")
src_txt = Path("/home/jupyter/HocTap/son/data_train/UniMER-1M/train.txt")
# print(f"Thư mục ảnh tồn tại: {src_img_dir.exists()}")
# print(f"File text tồn tại: {src_txt.exists()}")

In [5]:
base_out = Path("/home/jupyter/HocTap/son/Nhom11_UniMERNet/data/Images_500k")
out_train_img = base_out / "train" / "images"
out_val_img = base_out / "val" / "images"
out_train_txt = base_out / "train" / "train.txt"
out_val_txt = base_out / "val" / "val.txt"
out_train_map = base_out / "train_mapping.csv"
out_val_map = base_out / "val_mapping.csv"

In [7]:
# print(f"File text tồn tại: {base_out.exists()}")

File text tồn tại: True


In [10]:
out_train_img.mkdir(parents=True, exist_ok=True)
out_val_img.mkdir(parents=True, exist_ok=True)

In [11]:
# ====== config ======
TRAIN_SIZE = 500000
VAL_SIZE = 50000
TOTAL_SIZE = TRAIN_SIZE + VAL_SIZE
SEED = 42
random.seed(SEED)

In [12]:
# ====== read label ======
labels = src_txt.read_text(encoding="utf-8").splitlines()
print("Total labels:", len(labels))

all_imgs = list(src_img_dir.glob("*.png"))
print("Total image files found:", len(all_imgs))

Total labels: 1061791
Total image files found: 986122


In [13]:
valid_imgs = []
for p in all_imgs:
    try:
        idx = int(p.stem)
        if 0 <= idx < len(labels):
            valid_imgs.append((p, idx))
    except ValueError:
        pass

print("Valid images matched to train.txt:", len(valid_imgs))

if len(valid_imgs) < TOTAL_SIZE:
    raise ValueError(
        f"Không đủ dữ liệu hợp lệ. Cần {TOTAL_SIZE} mẫu nhưng chỉ có {len(valid_imgs)} mẫu."
    )

Valid images matched to train.txt: 986122


In [14]:
# ====== shuffle và chia tập ======
random.shuffle(valid_imgs)
selected = valid_imgs[:TOTAL_SIZE]

train_samples = selected[:TRAIN_SIZE]
val_samples = selected[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]

print("Train samples:", len(train_samples))
print("Val samples:", len(val_samples))

Train samples: 500000
Val samples: 50000


In [ ]:
# ====== xuất train / val ======
export_split(train_samples, out_train_img, out_train_txt, out_train_map)
export_split(val_samples, out_val_img, out_val_txt, out_val_map)

print("\nDone.")
print("Train dir:", out_train_img)
print("Val dir:", out_val_img)
print(f"Summary: train={TRAIN_SIZE}, val={VAL_SIZE}, total={TOTAL_SIZE}")

In [16]:
print("Train dir:", out_train_img)
print("Val dir:", out_val_img)
print(f"Summary: train={TRAIN_SIZE}, val={VAL_SIZE}, total={TOTAL_SIZE}")

Train dir: /home/jupyter/HocTap/son/Nhom11_UniMERNet/data/Images_500k/train/images
Val dir: /home/jupyter/HocTap/son/Nhom11_UniMERNet/data/Images_500k/val/images
Summary: train=500000, val=50000, total=550000


In [21]:
from pathlib import Path

# ====== cấu hình ======
img_dir = Path("/home/jupyter/HocTap/son/Nhom11_UniMERNet/data/Images_500k/val/images")
txt_path = Path("/home/jupyter/HocTap/son/Nhom11_UniMERNet/data/Images_500k/val/val.txt")

# ====== đọc label ======
labels = txt_path.read_text(encoding="utf-8").splitlines()
print(f"Total labels in txt: {len(labels)}")

# ====== đọc toàn bộ ảnh ======
img_files = sorted(img_dir.glob("*.png"))
print(f"Total images in folder: {len(img_files)}")

valid_count = 0
bad_name_files = []
out_of_range_files = []
matched_indices = set()

for img_path in img_files:
    try:
        idx = int(img_path.stem)   # ví dụ 0000123.png -> 123
    except ValueError:
        bad_name_files.append(img_path.name)
        continue

    if idx < 0 or idx >= len(labels):
        out_of_range_files.append((img_path.name, idx))
        continue

    # nếu tới đây nghĩa là ảnh có thể map tới label[idx]
    matched_indices.add(idx)
    valid_count += 1

# ====== tìm label không có ảnh tương ứng ======
missing_images = []
for i in range(len(labels)):
    if i not in matched_indices:
        missing_images.append(i)

# ====== báo cáo ======
print("\n===== CHECK RESULT =====")
print(f"Valid matched image-label pairs: {valid_count}")

if len(img_files) == len(labels) and len(bad_name_files) == 0 and len(out_of_range_files) == 0 and len(missing_images) == 0:
    print("✅ Ảnh và label khớp hoàn toàn.")
else:
    print("❌ Phát hiện dữ liệu chưa khớp hoàn toàn.")

print(f"\nBad image names (không parse được số): {len(bad_name_files)}")
if bad_name_files[:20]:
    print("Ví dụ:", bad_name_files[:20])

print(f"\nImages with index out of range: {len(out_of_range_files)}")
if out_of_range_files[:20]:
    print("Ví dụ:", out_of_range_files[:20])

print(f"\nLabels without corresponding image: {len(missing_images)}")
if missing_images[:20]:
    print("Ví dụ index thiếu ảnh:", missing_images[:20])

# ====== in thử vài cặp khớp đầu tiên ======
print("\n===== SAMPLE MATCHES =====")
shown = 0
for img_path in img_files:
    try:
        idx = int(img_path.stem)
    except ValueError:
        continue
    if 0 <= idx < len(labels):
        print(f"{img_path.name}  <-->  labels[{idx}] = {labels[idx][:150]}")
        shown += 1
        if shown >= 10:
            break


Total labels in txt: 50000
Total images in folder: 50000

===== CHECK RESULT =====
Valid matched image-label pairs: 50000
✅ Ảnh và label khớp hoàn toàn.

Bad image names (không parse được số): 0

Images with index out of range: 0

Labels without corresponding image: 0

===== SAMPLE MATCHES =====
0000000.png  <-->  labels[0] = \frac { 2 ( n + \alpha ) } { n + 1 }
0000001.png  <-->  labels[1] = \epsilon = 1 0
0000002.png  <-->  labels[2] = \begin{array} { r } { \overline { { c } } _ { n } ^ { c o r r } = \frac { \sum _ { w = 1 } ^ { W } \left( \sum _ { p _ { w } = 1 } ^ { P _ { w } } \rh
0000003.png  <-->  labels[3] = \frac { \partial \phi } { \partial t } + \nabla \cdot \vec { f } = \frac { \partial } { \partial x } \left( D _ { x } ^ { \ast } \frac { \partial f _ 
0000004.png  <-->  labels[4] = N = 2 , q = 3 , p = 0 , m _ { 0 } = 1 , n _ { 0 } = 0
0000005.png  <-->  labels[5] = x - y
0000006.png  <-->  labels[6] = \boldsymbol { V } _ { p } ^ { \mathrm { ~ 2 ~ h ~ 1 ~ p ~ } }
0000007.pn

In [ ]:
%pip install "setuptools<70.0.0"

#!sudo apt-get install -y libmagickwand-dev
%cd /home/jupyter/HocTap/son/Nhom11_UniMERNet/models/
!git clone https://huggingface.co/wanderkid/unimernet

Note: you may need to restart the kernel to use updated packages.
/home/jupyter/HocTap/son/Nhom11_UniMERNet/models
Cloning into 'unimernet'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20 (delta 5), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (20/20), 598.88 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [4]:
%pip install -U "unimernet[full]"

Note: you may need to restart the kernel to use updated packages.


In [18]:
%cd /home/jupyter/HocTap/son/Nhom11_UniMERNet
!/home/jupyter/venvs/lab_env/bin/python train.py --cfg-path /home/jupyter/HocTap/son/Nhom11_UniMERNet/configs/train/train_100k.yaml

/home/jupyter/HocTap/son/Nhom11_UniMERNet
/home/jupyter/venvs/lab_env/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Not using distributed mode
2026-03-27 18:42:32,906 [INFO] 
=====  Running Parameters    =====
2026-03-27 18:42:32,907 [INFO] {
    "accum_grad_iters": 1,
    "amp": true,
    "batch_size_eval": 4,
    "batch_size_train": 4,
    "device": "cuda",
    "dist_url": "env://",
    "distributed": false,
    "distributed_type": "ddp",
    "evaluate": false,
    "generate_cfg": {
        "temperature": 0.0
    },
    "init_lr": 5e-05,
    "iters_per_inner_epoch": 25,
    "lr_sched": "linear_warmup_cosine_lr",
    "max_iters": 200,
    "milestone": [
        1
    ],
    "min_lr": 1e-06,
    "num_workers": 2,
    "output_dir

In [ ]:
# File "/home/jupyter/HocTap/son/Nhom11_UniMERNet/unimernet/tasks/unimernet_train.py", line 138, in _report_metrics
 #   bleu = evaluate.load("bleu", keep_in_memory=True, experiment_id=random.randint(1, 1e8))
                                                                    ^^^^^^^^^^^^^^^^^^^^^^
  #File "/usr/lib/python3.12/random.py", line 336, in randint
   # return self.randrange(a, b+1)
           ^^^^^^^^^^^^^^^^^^^^^^
  #File "/usr/lib/python3.12/random.py", line 312, in randrange
  #  istop = _index(stop)
#TypeError: 'float' object cannot be interpreted as an integer